# Latent SOH Forecast Walkthrough

This notebook uses the existing `latent_soh` implementation as the label-building step for forecasting.

End-to-end flow:

1. Build or load latent SOH event tables with the existing Kalman-based `latent_soh` pipeline
2. Build the event forecasting dataset from `latent_soh_smooth_pct`
3. Train the current `soh_forecast` benchmark models
4. Inspect test and out-of-distribution performance
5. Reconstruct predicted next SOH and compare it to the latent target

This answers the practical question: if we trust the smoothed latent SOH more than raw observed SOH, how well can we forecast it?


## Setup

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "ml_workspace").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml_workspace.latent_soh.build_latent_soh import build_latent_soh_labels

PLANES = ["166", "192"]
RUN_LATENT_PIPELINE = False
RUN_EVENT_DATASET_BUILD = False
RUN_FORECAST_TRAINING = False

LATENT_RT_PROFILE = "balanced"
LATENT_Q_DAY_SIGMA_PCT = 0.05
LATENT_COMPARE_BACKEND = True
MAX_GAP_DAYS = 30.0

LATENT_ROOT = REPO_ROOT / "ml_workspace" / "latent_soh" / "output"
FORECAST_ROOT = REPO_ROOT / "ml_workspace" / "soh_forecast" / "output"
DATASET_PATH = FORECAST_ROOT / "event_dataset.csv"
METRICS_PATH = FORECAST_ROOT / "model_metrics.json"
MODELS_DIR = FORECAST_ROOT / "models"
SPEC_PATH = REPO_ROOT / "ml_workspace" / "battery_specs.yaml"
TIMESERIES_PATH = (
    REPO_ROOT / "data" / "event_timeseries_corrected.parquet"
    if (REPO_ROOT / "data" / "event_timeseries_corrected.parquet").exists()
    else REPO_ROOT / "data" / "event_timeseries.parquet"
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

print("Repo root:", REPO_ROOT)
print("Latent root:", LATENT_ROOT)
print("Forecast root:", FORECAST_ROOT)
print("Timeseries:", TIMESERIES_PATH)


## 1. Build or load latent SOH outputs

In [ ]:
if RUN_LATENT_PIPELINE:
    latent_results = {}
    for plane_id in PLANES:
        compare_backend = LATENT_COMPARE_BACKEND if plane_id == "166" else False
        latent_results[plane_id] = build_latent_soh_labels(
            plane_id=plane_id,
            timeseries_path=TIMESERIES_PATH,
            spec_path=SPEC_PATH,
            output_dir=LATENT_ROOT / f"plane_{plane_id}",
            q_day_sigma_pct=LATENT_Q_DAY_SIGMA_PCT,
            compare_backend=compare_backend,
            rt_profile=LATENT_RT_PROFILE,
        )
    print(json.dumps(latent_results, indent=2, default=str))
else:
    print("Using existing latent SOH outputs.")


In [ ]:
latent_tables = []
latent_summaries = []
for plane_id in PLANES:
    latent_path = LATENT_ROOT / f"plane_{plane_id}" / "latent_soh_event_table.csv"
    summary_path = LATENT_ROOT / f"plane_{plane_id}" / "diagnostics" / "smoother_summary.json"
    df = pd.read_csv(latent_path, parse_dates=["event_datetime"])
    summary = json.loads(summary_path.read_text())
    latent_tables.append(df)
    latent_summaries.append(
        {
            "plane_id": plane_id,
            "rows": len(df),
            "rt_profile": summary["rt_profile"],
            "q_day_sigma_pct": summary["q_day_sigma_pct"],
            "n_events_dropped_missing_observed_soh": summary["n_events_dropped_missing_observed_soh"],
        }
    )
latent_all = pd.concat(latent_tables, ignore_index=True)
display(pd.DataFrame(latent_summaries))
display(
    latent_all.groupby(["plane_id", "battery_id"], as_index=False).agg(
        n_events=("flight_id", "count"),
        observed_soh_min=("observed_soh_pct", "min"),
        observed_soh_max=("observed_soh_pct", "max"),
        latent_soh_min=("latent_soh_smooth_pct", "min"),
        latent_soh_max=("latent_soh_smooth_pct", "max"),
    ).round(3)
)


## 2. Visual check: observed SOH vs latent SOH

In [ ]:
fig, axes = plt.subplots(len(PLANES), 2, figsize=(14, 4 * len(PLANES)), sharex=False)
axes = np.atleast_2d(axes)
for row_idx, plane_id in enumerate(PLANES):
    plane_df = latent_all[latent_all["plane_id"].astype(str) == plane_id].copy()
    for col_idx, (battery_id, group) in enumerate(plane_df.groupby("battery_id", sort=True)):
        ax = axes[row_idx, col_idx]
        g = group.sort_values(["event_datetime", "flight_id"])
        ax.plot(g["event_datetime"], g["observed_soh_pct"], color="0.75", linewidth=1.0, label="Observed SOH")
        ax.plot(g["event_datetime"], g["latent_soh_smooth_pct"], color="#d62728", linewidth=1.8, label="Latent SOH")
        ax.set_title(f"Plane {plane_id} Battery {battery_id}")
        ax.set_ylabel("SOH (%)")
        ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()


## 3. Build the forecast dataset from latent SOH

In [ ]:
if RUN_EVENT_DATASET_BUILD:
    cmd = [
        str(REPO_ROOT / ".venv" / "bin" / "python"),
        "-m",
        "ml_workspace.soh_forecast.build_event_dataset",
        "--planes",
        ",".join(PLANES),
        "--latent-root",
        str(LATENT_ROOT),
        "--output-root",
        str(FORECAST_ROOT),
        "--max-gap-days",
        str(MAX_GAP_DAYS),
    ]
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)
else:
    print("Using existing event_dataset.csv")


In [ ]:
dataset_df = pd.read_csv(DATASET_PATH, parse_dates=["event_datetime", "next_event_datetime", "next_flight_datetime"])
dataset_df["plane_id"] = dataset_df["plane_id"].astype(str)
dataset_df["battery_id"] = pd.to_numeric(dataset_df["battery_id"], errors="coerce")

print("Dataset rows:", len(dataset_df))
display(
    dataset_df.groupby("plane_id", as_index=False).agg(
        rows=("event_datetime", "count"),
        rows_with_next=("has_next", "sum"),
        latent_soh_mean=("latent_soh_smooth_pct", "mean"),
        delta_soh_mean=("delta_soh", "mean"),
        delta_soh_std=("delta_soh", "std"),
    ).round(4)
)
display(
    dataset_df[
        [
            "plane_id",
            "battery_id",
            "event_datetime",
            "event_type",
            "latent_soh_smooth_pct",
            "next_soh",
            "delta_soh",
            "delta_days",
            "cumulative_efc",
        ]
    ].head(12).round(4)
)


## 4. Train forecast models on the latent target

In [ ]:
if RUN_FORECAST_TRAINING:
    cmd = [
        str(REPO_ROOT / ".venv" / "bin" / "python"),
        "-m",
        "ml_workspace.soh_forecast.train_degradation_models",
        "--dataset-path",
        str(DATASET_PATH),
        "--output-root",
        str(FORECAST_ROOT),
    ]
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)
else:
    print("Using existing trained forecast outputs.")


In [ ]:
metrics = json.loads(METRICS_PATH.read_text())

baseline_df = pd.DataFrame(metrics["baselines"])
model_df = pd.DataFrame(metrics["models"])

if "test_metrics" in model_df.columns:
    model_df["test_mae"] = model_df["test_metrics"].apply(lambda x: x.get("mae") if isinstance(x, dict) else np.nan)
    model_df["test_rmse"] = model_df["test_metrics"].apply(lambda x: x.get("rmse") if isinstance(x, dict) else np.nan)
    model_df["test_r2"] = model_df["test_metrics"].apply(lambda x: x.get("r2") if isinstance(x, dict) else np.nan)
    model_df["ood_mae"] = model_df["ood_metrics"].apply(lambda x: x.get("mae") if isinstance(x, dict) else np.nan)
    model_df["ood_r2"] = model_df["ood_metrics"].apply(lambda x: x.get("r2") if isinstance(x, dict) else np.nan)

if "metrics" in baseline_df.columns:
    baseline_df["test_mae"] = baseline_df["metrics"].apply(lambda x: x.get("mae") if isinstance(x, dict) else np.nan)
    baseline_df["test_rmse"] = baseline_df["metrics"].apply(lambda x: x.get("rmse") if isinstance(x, dict) else np.nan)
    baseline_df["test_r2"] = baseline_df["metrics"].apply(lambda x: x.get("r2") if isinstance(x, dict) else np.nan)

display(baseline_df[["model_name", "test_mae", "test_rmse", "test_r2"]].sort_values("test_mae").round(4))
display(model_df[["model_name", "test_mae", "test_rmse", "test_r2", "ood_mae", "ood_r2"]].sort_values("test_mae").round(4))


## 5. Recreate the test split and generate predictions

In [ ]:
from ml_workspace.soh_forecast.train_degradation_models import _build_fe2_mission_df, _time_split, _build_feature_frame as train_build_feature_frame

work = dataset_df.loc[dataset_df["has_next"] == True].copy()  # noqa: E712
target_name = metrics["config"]["target_name"]
mission_df = _build_fe2_mission_df(work, target_name=target_name, nominal_capacity_ah=metrics["config"]["nominal_capacity_ah"])
mission_df = mission_df.loc[mission_df[target_name].notna()].copy()

plane_166 = mission_df.loc[mission_df["plane_id"] == "166"].copy()
plane_192 = mission_df.loc[mission_df["plane_id"] == "192"].copy()
train_df, valid_df, test_df = _time_split(plane_166, metrics["config"]["train_frac"], metrics["config"]["valid_frac"])

print("Train / valid / test sizes:", len(train_df), len(valid_df), len(test_df))
print("Plane 192 rows:", len(plane_192))


In [ ]:
def load_model_payload(model_name: str):
    return joblib.load(MODELS_DIR / f"{model_name}.joblib")


def predict_with_payload(payload, df: pd.DataFrame) -> np.ndarray:
    bundle = payload["bundle"]
    scaler = payload["scaler"]
    model = payload["model"]
    feat_names = bundle["feature_names"]
    medians = pd.Series(bundle["feature_medians"])
    numeric_cols = [c for c in feat_names if not c.startswith("event_type_")]
    dummy_cols = [c for c in feat_names if c.startswith("event_type_")]
    X, _, _ = train_build_feature_frame(df, numeric_cols, "event_type", dummy_cols, medians)
    X_scaled = scaler.transform(X)
    return model.predict(X_scaled)


model_names = ["throughput_linear", "elastic_net", "hist_gbdt", "arx_ridge"]
pred_df = test_df[
    [
        "plane_id",
        "battery_id",
        "event_datetime",
        "mission_end",
        "event_type",
        "latent_soh_smooth_pct",
        target_name,
    ]
].copy()

for model_name in model_names:
    payload = load_model_payload(model_name)
    pred_df[f"pred_{model_name}"] = predict_with_payload(payload, test_df)

pred_df["actual_next_latent_soh"] = pred_df["latent_soh_smooth_pct"] + pred_df[target_name]
for model_name in model_names:
    pred_df[f"pred_next_soh_{model_name}"] = pred_df["latent_soh_smooth_pct"] + pred_df[f"pred_{model_name}"]

display(pred_df.head(12).round(4))


## 6. Forecast performance plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, color in [
    ("throughput_linear", "#ff7f0e"),
    ("elastic_net", "#1f77b4"),
    ("hist_gbdt", "#2ca02c"),
    ("arx_ridge", "#d62728"),
]:
    axes[0].scatter(
        pred_df["actual_next_latent_soh"],
        pred_df[f"pred_next_soh_{model_name}"],
        alpha=0.65,
        s=20,
        label=model_name,
        color=color,
    )
lo = min(pred_df["actual_next_latent_soh"].min(), pred_df[[c for c in pred_df.columns if c.startswith("pred_next_soh_")]].min().min())
hi = max(pred_df["actual_next_latent_soh"].max(), pred_df[[c for c in pred_df.columns if c.startswith("pred_next_soh_")]].max().max())
axes[0].plot([lo, hi], [lo, hi], linestyle="--", color="0.4")
axes[0].set_title("Test set: predicted next latent SOH vs actual")
axes[0].set_xlabel("Actual next latent SOH (%)")
axes[0].set_ylabel("Predicted next latent SOH (%)")
axes[0].legend(loc="best")
axes[0].grid(alpha=0.25)

plot_df = pred_df.sort_values("event_datetime").copy()
axes[1].plot(plot_df["event_datetime"], plot_df["actual_next_latent_soh"], color="0.15", linewidth=2.0, label="Actual next latent SOH")
axes[1].plot(plot_df["event_datetime"], plot_df["pred_next_soh_elastic_net"], color="#1f77b4", linewidth=1.7, label="ElasticNet")
axes[1].plot(plot_df["event_datetime"], plot_df["pred_next_soh_arx_ridge"], color="#d62728", linewidth=1.7, label="ARX Ridge")
axes[1].plot(plot_df["event_datetime"], plot_df["pred_next_soh_hist_gbdt"], color="#2ca02c", linewidth=1.7, label="HistGBDT")
axes[1].set_title("Chronological next-SOH forecast on plane 166 test split")
axes[1].set_xlabel("Event datetime")
axes[1].set_ylabel("SOH (%)")
axes[1].legend(loc="best")
axes[1].grid(alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()


## 7. Forecast error by model

In [ ]:
error_rows = []
for model_name in model_names:
    level_err = pred_df[f"pred_next_soh_{model_name}"] - pred_df["actual_next_latent_soh"]
    delta_err = pred_df[f"pred_{model_name}"] - pred_df[target_name]
    error_rows.append(
        {
            "model_name": model_name,
            "next_soh_mae": np.mean(np.abs(level_err)),
            "next_soh_rmse": np.sqrt(np.mean(level_err ** 2)),
            "delta_mae": np.mean(np.abs(delta_err)),
            "delta_rmse": np.sqrt(np.mean(delta_err ** 2)),
            "delta_bias": np.mean(delta_err),
        }
    )
error_df = pd.DataFrame(error_rows).sort_values("next_soh_mae")
display(error_df.round(4))


## 8. Out-of-distribution check on plane 192

In [ ]:
ood_rows = []
for model_name in model_names:
    payload = load_model_payload(model_name)
    ood_pred = predict_with_payload(payload, plane_192)
    ood_level = plane_192["latent_soh_smooth_pct"].to_numpy(dtype=float) + ood_pred
    actual_level = plane_192["latent_soh_smooth_pct"].to_numpy(dtype=float) + plane_192[target_name].to_numpy(dtype=float)
    err = ood_level - actual_level
    ood_rows.append(
        {
            "model_name": model_name,
            "ood_next_soh_mae": np.mean(np.abs(err)),
            "ood_next_soh_rmse": np.sqrt(np.mean(err ** 2)),
        }
    )
ood_df = pd.DataFrame(ood_rows).sort_values("ood_next_soh_mae")
display(ood_df.round(4))


## 9. Interpretation

In [ ]:
best_test = error_df.sort_values("next_soh_mae").iloc[0]
best_ood = ood_df.sort_values("ood_next_soh_mae").iloc[0]

print("Interpretation:")
print(f"- Best in-distribution next-SOH forecast model: {best_test['model_name']} with MAE {best_test['next_soh_mae']:.4f}.")
print(f"- Best out-of-distribution model on plane 192: {best_ood['model_name']} with MAE {best_ood['ood_next_soh_mae']:.4f}.")
print("- These forecasts are based on the latent SOH target produced by the existing Kalman-style `latent_soh` pipeline, not the raw observed SOH spikes.")
print("- If the latent-target forecast errors are materially smaller or more stable than raw-SOH forecasting, that supports using latent SOH as the forecasting label.")
print("- ARX Ridge usually benefits if latent SOH is smooth enough that short target-history lags become informative.")
